## Project Introduction

### Project Goals
- Parse Wikipedia article: 'List of highest-grossing films'
- Extract structured dataset (film title, year, worldwide gross, etc.)
- Clean and normalize extracted records
- Persist data in SQLite for reproducible querying
- Export final dataset to JSON for static website integration

### Tools and Libraries
- `requests` / `httpx` (HTTP retrieval)
- `BeautifulSoup` (`bs4`) or `lxml` (HTML parsing)
- `pandas` (tabular cleaning, transformation)
- `sqlite3` (database persistence)
- `json` (data interchange)
- optional: `sqlalchemy`, `python-dotenv`

### Pipeline Steps
1. Data acquisition
2. HTML parsing and extraction
3. Data cleaning and normalization
4. SQLite storage
5. JSON export

### Short Explanation of Each Step
- Data acquisition: Download the Wikipedia page and cache immutable HTML for repeatable experiments.
- HTML parsing and extraction: Identify highest-grossing film tables, capture row-level fields, and assemble raw data into a DataFrame-like structure.
- Data cleaning and normalization: Convert financial and date fields to numeric types, standardize names, handle missing values, and de-duplicate entries.
- SQLite storage: Design and populate a schema in an SQLite database for robust, indexable storage and analytics.
- JSON export: Serialize the cleaned and indexed dataset into a stable JSON format suitable for static website consumption.

## Step 1: Imports and Configuration

In [10]:
import re
import time
import sqlite3
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup
import json

# Configuration
BASE_URL = "https://en.wikipedia.org"
START_URL = "https://en.wikipedia.org/wiki/List_of_highest-grossing_films"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/146.0.0.0 Safari/537.36"
    )
}

## Step 2: Utility Functions (Data Cleaning & Parsing)

In [11]:
def clean_text(text: str) -> str:
    """Removes citations, extra spaces, special characters, and noise from the string."""
    if text is None:
        return ""
    text = re.sub(r"\[[^\]]*\]", "", text)          # [1], [a], [note 2]
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    text = text.replace("†", "").strip()
    return text


def parse_money(text: str):
    """
    Converts strings like '$2,923,710,708' or '$2.924 billion'
    to float. Returns None if conversion fails.
    """
    text = clean_text(text).lower()
    if not text:
        return None

    # Remove footnote/citation markers like "F8", "A1", etc. at the beginning
    # Pattern: letter(s) + digits at start, followed by space/$ -> remove it
    text = re.sub(r'^[a-z]+\d*\s+', '', text)
    
    # If text starts with $ or contains it, extract from $ onwards
    if '$' in text:
        # Find position of $ and take everything after it
        dollar_pos = text.find('$')
        text = text[dollar_pos + 1:]  # Remove $ and everything before it
    
    # Keep only digits, dots, commas, and keywords billion/million
    if "billion" in text:
        m = re.search(r"([\d,.]+)", text)
        if not m:
            return None
        num = float(m.group(1).replace(",", ""))
        return num * 1_000_000_000

    if "million" in text:
        m = re.search(r"([\d,.]+)", text)
        if not m:
            return None
        num = float(m.group(1).replace(",", ""))
        return num * 1_000_000

    # Standard format with dollar values
    nums = re.sub(r"[^\d.]", "", text.replace(",", ""))
    if not nums:
        return None
    try:
        return float(nums)
    except ValueError:
        return None


def get_soup(url: str) -> BeautifulSoup:
    """Downloads page and returns BeautifulSoup."""
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")

## Step 3: HTML Parsing and Extraction Functions

In [12]:
def find_main_table(soup: BeautifulSoup):
    """
    Finds the main highest-grossing films table.
    It has headers: Rank / Peak / Title / Worldwide gross / Year / Ref.
    """
    for table in soup.find_all("table", class_="wikitable"):
        headers = [clean_text(th.get_text(" ", strip=True)) for th in table.find_all("th")]
        header_blob = " | ".join(headers)
        if "Title" in header_blob and "Worldwide gross" in header_blob and "Year" in header_blob:
            return table
    return None


def extract_title_link_and_basic_fields(row):
    """
    Extracts title, year, gross, and link from table row.
    """
    cells = row.find_all(["td", "th"])
    if len(cells) < 5:
        return None

    # In table: Rank | Peak | Title | Worldwide gross | Year | Ref
    title_cell = cells[2]
    gross_cell = cells[3]
    year_cell = cells[4]

    a = title_cell.find("a", href=True)
    href = urljoin(BASE_URL, a["href"]) if a else None
    title = clean_text(title_cell.get_text(" ", strip=True))
    gross = clean_text(gross_cell.get_text(" ", strip=True))
    year_text = clean_text(year_cell.get_text(" ", strip=True))

    # sometimes title contains citations/markers, remove them
    title = re.sub(r"\s*†\s*$", "", title).strip()

    try:
        year = int(re.search(r"\d{4}", year_text).group())
    except Exception:
        year = None

    return {
        "title": title,
        "release_year": year,
        "box_office_raw": gross,
        "url": href,
    }


def normalize_text_with_separators(text: str) -> str:
    """
    Normalizes text by separating merged names/titles with commas.
    IMPORTANT: Does not split regular names (James Cameron), initials (J.J. Abrams),
    or multi-word countries (United States)!
    Only splits: camelCase (RussoJoe), multiple DIFFERENT countries (USA China, New Zealand Germany).
    """
    if not text:
        return text
    
    # Normalize spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # If there's already a comma, it's multiple values - just clean up
    if ',' in text:
        # Handle special case: single initial incorrectly separated (e.g., "F., Gary, Gray" -> "F. Gary Gray")
        # Pattern: Single letter + dot + comma + space + word (not an initial)
        if re.match(r'^[A-Z]\.,\s+', text):
            # Split by comma
            parts = [p.strip() for p in text.split(',')]
            # First part is initial (e.g., "F.")
            # If there are 2+ other parts and they look like a name, combine them
            if len(parts) >= 3 and not parts[1].endswith('.'):
                # Combine: initial + space + remaining parts (space-separated)
                return f"{parts[0]} {' '.join(parts[1:])}"
        
        text = re.sub(r',\s+', ', ', text)
        text = re.sub(r'\s+,', ',', text)
        return text.strip()
    
    # ========== PROTECT FROM INITIALS (PRIORITY 0) ==========
    # If text contains initials like "J.J. Abrams", "K.D. Lang", "A.A. Milne"
    # or "J., J., Abrams" (after list join)
    # Make sure we don't split them and fix the format
    # Pattern: one letter + dot + (comma/space) + one letter + dot
    has_initials = bool(re.search(r'[A-Z]\.[\s,]+[A-Z]\.', text))
    
    if has_initials:
        # Fix format "J., J., Abrams" -> "J.J. Abrams"
        # Remove commas and extra spaces between initials
        text = re.sub(r'([A-Z])\.,\s+', r'\1.', text)
        text = re.sub(r'([A-Z])\.\s+([A-Z])\.', r'\1.\2.', text)
        return text.strip()
    
    # ========== PROCESS camelCase (PRIORITY 1) =========="
    if re.search(r'[a-z][A-Z]', text):
        text = re.sub(r'([a-z])([A-Z])', r'\1, \2', text)
        return text.strip()
    
    # ========== PROCESS COUNTRIES AND REGIONS (PRIORITY 2) ==========
    # List of multi-word regions that should stay together
    known_multiword_regions = [
        'United States', 'United Kingdom', 'New Zealand', 'Costa Rica',
        'Saudi Arabia', 'South Africa', 'South Korea', 'North Korea',
        'Hong Kong', 'Puerto Rico', 'Sierra Leone', 'Papua New Guinea',
        'Equatorial Guinea', 'Dominican Republic', 'Trinidad and Tobago',
        'Antigua and Barbuda', 'Saint Kitts and Nevis', 'Bosnia and Herzegovina',
        'Czech Republic', 'Marshall Islands', 'Solomon Islands', 'Sri Lanka',
        'Cayman Islands', 'British Virgin Islands', 'Northern Mariana Islands',
    ]
    
    # IMPORTANT: If starts with single initial (like "F. Gary Gray"), do NOT split!
    # This is a single person name, not multiple values
    if re.match(r'^[A-Z]\.\s+[A-Z]', text):
        # Starts with initial + space + capital letter = name pattern
        # Don't split, just return as is
        return text
    
    # Replace multi-word regions with placeholders
    placeholders = {}
    placeholder_text = text
    for i, region in enumerate(known_multiword_regions):
        placeholder = f"__REGION_{i}__"
        if region in placeholder_text:
            placeholders[placeholder] = region
            placeholder_text = placeholder_text.replace(region, placeholder)
    
    # Now check if placeholder_text has multiple words starting with capital letters
    words = placeholder_text.split()
    
    # If there are placeholders, it's probably a list of countries
    has_placeholders = any(w.startswith("__REGION_") for w in words)
    
    # If after placeholder comes another word (country) - split them
    if has_placeholders or len(words) > 2:
        # If many words (>2) or there are placeholders - it's a list of countries
        # Split with comma
        if len(words) > 2 or (has_placeholders and len(words) == 2):
            result_words = []
            for word in words:
                if word.startswith("__REGION_"):
                    result_words.append(placeholders[word])
                else:
                    result_words.append(word)
            return ', '.join(result_words)
    
    # If exactly 2 words and there's one placeholder and one regular word - these are two countries
    if len(words) == 2 and any(w.startswith("__REGION_") for w in words):
        result_words = []
        for word in words:
            if word.startswith("__REGION_"):
                result_words.append(placeholders[word])
            else:
                result_words.append(word)
        return ', '.join(result_words)
    
    # If all are placeholders or just words, restore them
    for placeholder, region in placeholders.items():
        placeholder_text = placeholder_text.replace(placeholder, region)
    
    # ========== PROCESS REGULAR TWO-WORD PHRASES ==========
    # If 2 words remain (like "James Cameron"), don't split
    # Since we already checked for camelCase and regions above
    
    # Final normalization
    result = placeholder_text
    result = re.sub(r',\s*,', ',', result)
    result = re.sub(r',\s*$', '', result).strip()
    
    return result


def get_infobox_value(soup: BeautifulSoup, labels):
    """
    Finds value in infobox by one of the field names.
    Separates authors and countries with comma + space.
    """
    infobox = soup.find("table", class_=lambda c: c and any(x in c.lower() for x in ["infobox", "vcard"]))
    
    if not infobox:
        return ""

    rows = infobox.find_all("tr")
    for row in rows:
        header_cell = row.find("th")
        value_cell = row.find("td")
        
        if not header_cell or not value_cell:
            continue

        header_text = clean_text(header_cell.get_text(" ", strip=True)).lower()
        
        match = False
        for label in labels:
            if label.lower() in header_text or header_text == label.lower():
                match = True
                break
        
        if not match:
            continue

        # Strategy 1: Find all links
        links = value_cell.find_all("a")
        if links:
            texts = [clean_text(link.get_text()) for link in links if clean_text(link.get_text())]
            texts = [t for t in texts if len(t) > 1 and t not in ["w", "v", "t", "e"]]
            if texts:
                result = ", ".join(texts)
                result = normalize_text_with_separators(result)
                return result
        
        # Strategy 2: Parse text manually
        raw_text = str(value_cell)
        raw_text = raw_text.replace("<br/>", " | ").replace("<br>", " | ").replace("<br />", " | ")
        
        clean = BeautifulSoup(raw_text, "html.parser").get_text(" ", strip=True)
        
        # Convert 'and' and '&' to comma
        clean = re.sub(r'\s+and\s+', ', ', clean, flags=re.IGNORECASE)
        clean = re.sub(r'\s*&\s*', ', ', clean, flags=re.IGNORECASE)
        
        # Split by comma, semicolon, and pipe
        parts = re.split(r'[\s]*[|,;]\s*', clean)
        parts = [clean_text(p) for p in parts if clean_text(p) and len(clean_text(p)) > 1]
        
        result = ", ".join(parts) if parts else clean
        result = normalize_text_with_separators(result)
        
        return result if result else ""

    return ""


def parse_film_page(film_url: str):
    """
    Fetches film page and extracts director and country from infobox.
    """
    try:
        soup = get_soup(film_url)
        
        director = get_infobox_value(soup, ["Directed by", "Director", "Directors"])
        country = get_infobox_value(soup, ["Country", "Countries", "Country of origin"])

        return {
            "director": director if director else "",
            "country": country if country else "",
        }
    except Exception as e:
        print(f"⚠️  Error parsing {film_url}: {str(e)[:50]}")
        return {
            "director": "",
            "country": "",
        }


## Step 4: Main Parsing Execution

In [13]:
def parse_highest_grossing_films():
    """
    Main parsing function:
    1) downloads main page;
    2) finds table;
    3) parses rows;
    4) for each film fetches film page and gets director/country;
    5) returns DataFrame.
    """
    soup = get_soup(START_URL)
    table = find_main_table(soup)

    if table is None:
        raise RuntimeError("Failed to find main highest-grossing films table.")

    films = []

    rows = table.find_all("tr")
    for row in rows[1:]:
        basic = extract_title_link_and_basic_fields(row)
        if not basic:
            continue

        # skip rows without film link
        if not basic["url"]:
            continue

        try:
            extra = parse_film_page(basic["url"])
            time.sleep(0.5)  # throttle to not overload Wikipedia
        except Exception as e:
            extra = {"director": "", "country": ""}
            print(f"Failed to load {basic['title']}: {e}")

        film = {
            "title": basic["title"],
            "release_year": basic["release_year"],
            "director": extra["director"],
            "box_office": parse_money(basic["box_office_raw"]),
            "country": extra["country"],
            "source_url": basic["url"],
        }
        films.append(film)

    df = pd.DataFrame(films)

    # Remove rows with critical missing values
    df = df.dropna(subset=["title", "release_year"])
    df = df[df["title"].str.len() > 0].reset_index(drop=True)

    return df


# Run parsing
print("⏳ Downloading and parsing Wikipedia...")
df_films = parse_highest_grossing_films()

print("\n✅ Parsing complete!")
print(f"Extracted {len(df_films)} films\n")
print("First 10 films:")
print(df_films.head(10))

⏳ Downloading and parsing Wikipedia...

✅ Parsing complete!
Extracted 50 films

First 10 films:
                          title  release_year                  director  \
0                        Avatar          2009             James Cameron   
1             Avengers: Endgame          2019  Anthony Russo, Joe Russo   
2      Avatar: The Way of Water          2022             James Cameron   
3                       Titanic          1997             James Cameron   
4                      Ne Zha 2          2025                    Jiaozi   
5  Star Wars: The Force Awakens          2015               J.J. Abrams   
6        Avengers: Infinity War          2018  Anthony Russo, Joe Russo   
7       Spider-Man: No Way Home          2021                 Jon Watts   
8                    Zootopia 2          2025  Jared Bush, Byron Howard   
9                  Inside Out 2          2024               Kelsey Mann   

     box_office                        country  \
0  2.923711e+09  United Stat

## Step 5: Store in SQLite Database

In [14]:
def save_to_sqlite(df: pd.DataFrame, db_path="films.db"):
    """
    Saves DataFrame to SQLite database.
    """
    print(f"💾 Saving {len(df)} films to database...")

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    # Create table
    cur.execute("""
        CREATE TABLE IF NOT EXISTS films (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            release_year INTEGER,
            director TEXT,
            box_office REAL,
            country TEXT
        )
    """)

    # Clear table before insertion
    cur.execute("DELETE FROM films")

    # Insert data
    inserted = 0
    for _, row in df.iterrows():
        try:
            cur.execute("""
                INSERT INTO films (title, release_year, director, box_office, country)
                VALUES (?, ?, ?, ?, ?)
            """, (
                str(row["title"]),
                int(row["release_year"]) if pd.notna(row["release_year"]) else None,
                str(row["director"]),
                float(row["box_office"]) if pd.notna(row["box_office"]) else None,
                str(row["country"]),
            ))
            inserted += 1
        except Exception as e:
            print(f"❌ Error inserting row: {e}")

    conn.commit()

    # Check what was actually saved
    cur.execute("SELECT COUNT(*) FROM films")
    count = cur.fetchone()[0]

    conn.close()

    print(f"✅ Inserted {inserted} rows into database")
    print(f"📊 Total rows in database: {count}\n")


# Execute save
save_to_sqlite(df_films)

💾 Saving 50 films to database...
✅ Inserted 50 rows into database
📊 Total rows in database: 50



## Step 6: Export to JSON

In [15]:
def export_db_to_json(db_path="films.db", json_path="films.json"):
    """
    Exports data from SQLite to JSON format for website.
    """
    print(f"📤 Exporting data from database to JSON...")

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    # Get all data from table
    cur.execute("SELECT title, release_year, director, box_office, country FROM films")
    rows = cur.fetchall()

    columns = ["title", "release_year", "director", "box_office", "country"]

    # Build list of dictionaries
    films = []
    for row in rows:
        film = dict(zip(columns, row))
        films.append(film)

    conn.close()

    # Save to JSON
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(films, f, indent=4, ensure_ascii=False)

    print(f"✅ Exported {len(films)} records")
    print(f"📁 JSON file saved: {json_path}\n")
    
    # Display sample of first record
    if films:
        print("Sample of first record in JSON:")
        print(json.dumps(films[0], indent=2, ensure_ascii=False))


# Execute export
export_db_to_json()

📤 Exporting data from database to JSON...
✅ Exported 50 records
📁 JSON file saved: films.json

Sample of first record in JSON:
{
  "title": "Avatar",
  "release_year": 2009,
  "director": "James Cameron",
  "box_office": 2923710708.0,
  "country": "United States, United Kingdom"
}
